# 03 · Morphological

Computes street width and plot area for each establishment using only the Overpass API.

- **Street width (m):** fetches all highways in the area, matches each shop to its nearest street segment, estimates width from `lanes × 3.5 m` with highway-type defaults.
- **Plot area (m²):** fetches all building polygons in the area, matches each shop to its nearest building, computes footprint area using the shoelace formula.

**Input:** `csv/00_base_data.csv`
**Output:** `csv/03_morphological.csv` — `osm_id`, `street_width_m`, `plot_area_m2`

In [ ]:
import requests
import pandas as pd
import math, time

df = pd.read_csv("csv/00_base_data.csv")
print(f"pandas {pd.__version__}")
print(f"Loaded {len(df)} records")

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
HEADERS = {"User-Agent": "osmnx-data-scraper/1.0 (research project)"}

# Study area params
origin_lat = (df["lat"].max() + df["lat"].min()) / 2
origin_lon = (df["lon"].max() + df["lon"].min()) / 2
RADIUS = 1800  # meters — covers shops + buffer

print(f"Origin: ({origin_lat:.5f}, {origin_lon:.5f})  Radius: {RADIUS} m")

In [ ]:
# ── Helpers ───────────────────────────────────────────
def overpass_json(query):
    for ep in OVERPASS_ENDPOINTS:
        try:
            r = requests.post(ep, data={"data": query}, headers=HEADERS, timeout=120)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"  x {ep}: {e}")
            time.sleep(3)
    raise RuntimeError("All Overpass endpoints failed.")


def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))


def polygon_area_m2(coords):
    """Area in m² of a polygon given as list of (lat, lon) tuples — shoelace formula."""
    if len(coords) < 3:
        return 0.0
    lat0 = sum(c[0] for c in coords) / len(coords)
    R = 6371000
    cos_lat = math.cos(math.radians(lat0))
    x = [R * math.radians(c[1]) * cos_lat for c in coords]
    y = [R * math.radians(c[0]) for c in coords]
    n = len(x)
    area = abs(sum(x[i]*y[(i+1)%n] - x[(i+1)%n]*y[i] for i in range(n))) / 2
    return round(area, 1)


# ── Street width ──────────────────────────────────────
DEFAULT_LANES = {
    "motorway": 4, "motorway_link": 2,
    "trunk": 4, "trunk_link": 2,
    "primary": 4, "primary_link": 2,
    "secondary": 2, "secondary_link": 1,
    "tertiary": 2, "tertiary_link": 1,
    "residential": 2, "living_street": 1,
    "unclassified": 2, "service": 1,
}
LANE_WIDTH_M = 3.5

query_streets = f"""
[out:json][timeout:90];
way["highway"](around:{RADIUS},{origin_lat},{origin_lon});
out center tags;
"""

print("Fetching streets...")
data = overpass_json(query_streets)

streets = []
for el in data.get("elements", []):
    if el.get("type") != "way": continue
    clat = el.get("center", {}).get("lat")
    clon = el.get("center", {}).get("lon")
    if clat is None: continue
    tags = el.get("tags", {})
    hw   = tags.get("highway", "residential")
    raw_lanes = tags.get("lanes")
    raw_width = tags.get("width")
    # Parse width
    if raw_width:
        try: width = round(float(str(raw_width).replace("m","").strip()), 1)
        except: width = None
    else:
        width = None
    # Estimate from lanes if width missing
    if width is None:
        try:   lanes = int(str(raw_lanes).split(";")[0])
        except: lanes = DEFAULT_LANES.get(hw, 2)
        width = round(lanes * LANE_WIDTH_M, 1)
    streets.append({"lat": clat, "lon": clon, "street_width_m": width})

print(f"{len(streets)} street segments loaded")

# For each shop find nearest street segment
s_lats = [s["lat"] for s in streets]
s_lons = [s["lon"] for s in streets]
s_widths = [s["street_width_m"] for s in streets]

def nearest_street_width(shop_lat, shop_lon):
    best_d, best_w = math.inf, None
    for i, (slat, slon) in enumerate(zip(s_lats, s_lons)):
        # Quick bbox filter
        if abs(slat - shop_lat) > 0.02 or abs(slon - shop_lon) > 0.03:
            continue
        d = haversine(shop_lat, shop_lon, slat, slon)
        if d < best_d:
            best_d, best_w = d, s_widths[i]
    return best_w

df["street_width_m"] = df.apply(
    lambda r: nearest_street_width(r["lat"], r["lon"]), axis=1
)
print(f"Street width fill: {df['street_width_m'].notna().sum()}/{len(df)}")

In [ ]:
# ── Plot area (building footprints via out geom) ──────
query_buildings = f"""
[out:json][timeout:120];
way["building"](around:{RADIUS},{origin_lat},{origin_lon});
out geom;
"""

print("Fetching buildings...")
data_b = overpass_json(query_buildings)

buildings = []
for el in data_b.get("elements", []):
    if el.get("type") != "way": continue
    geom = el.get("geometry", [])
    if len(geom) < 3: continue
    coords = [(g["lat"], g["lon"]) for g in geom]
    area   = polygon_area_m2(coords)
    if area <= 0: continue
    # Use centroid of polygon as reference point
    clat = sum(c[0] for c in coords) / len(coords)
    clon = sum(c[1] for c in coords) / len(coords)
    buildings.append({"lat": clat, "lon": clon, "area_m2": area})

print(f"{len(buildings)} buildings with geometry loaded")

b_lats  = [b["lat"]    for b in buildings]
b_lons  = [b["lon"]    for b in buildings]
b_areas = [b["area_m2"] for b in buildings]

def nearest_building_area(shop_lat, shop_lon):
    best_d, best_a = math.inf, None
    for i, (blat, blon) in enumerate(zip(b_lats, b_lons)):
        if abs(blat - shop_lat) > 0.02 or abs(blon - shop_lon) > 0.03:
            continue
        d = haversine(shop_lat, shop_lon, blat, blon)
        if d < best_d:
            best_d, best_a = d, b_areas[i]
    return best_a

df["plot_area_m2"] = df.apply(
    lambda r: nearest_building_area(r["lat"], r["lon"]), axis=1
)
print(f"Plot area fill: {df['plot_area_m2'].notna().sum()}/{len(df)}")

In [ ]:
df_out = df[["osm_id", "street_width_m", "plot_area_m2"]]
df_out.to_csv("csv/03_morphological.csv", index=False, encoding="utf-8")
print(f"Saved {len(df_out)} records to csv/03_morphological.csv")
print(df_out.describe().round(1))